# top

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from torchvision.utils import make_grid
from torchvision.datasets import ImageFolder
from torchsummary import summary
from PIL import Image
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix

%matplotlib inline

In [2]:
ds = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
diseases  = os.listdir(ds)
print(len(diseases))  #38 classes
# print(diseases )

38


# a

In [3]:
# ============================================================
# Phase 1: UA-ProtoNet PlantVillage Multi-Seed + Fixed Episodes
# ============================================================

import os
import random
import json
import glob
import contextlib
import csv
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

# ============================================================
# Shared Utilities & Seed Setting
# ============================================================

def set_seed(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def autocast_context(device: str):
    if str(device).startswith("cuda"):
        try:
            return torch.amp.autocast(device_type="cuda")
        except AttributeError:
            return torch.cuda.amp.autocast()
    return contextlib.nullcontext()


def make_grad_scaler(use_cuda: bool):
    if not use_cuda:
        return None
    try:
        return torch.amp.GradScaler(device="cuda")
    except AttributeError:
        return torch.cuda.amp.GradScaler()


def fmt(x, decimals: int = 4) -> str:
    if x is None:
        return "NA"
    try:
        if np.isnan(x):
            return "NA"
    except TypeError:
        pass
    return f"{x:.{decimals}f}"


# ============================================================
# ProtoNet Utilities
# ============================================================

def compute_prototypes(embeddings, labels, n_way):
    return torch.stack([embeddings[labels == c].mean(dim=0) for c in range(n_way)])


def euclidean_logits(query_embeddings, prototypes, temperature=1.0):
    return -torch.cdist(query_embeddings, prototypes, p=2) / temperature


# ============================================================
# Dataset helpers
# ============================================================

class ImageFolderWithIndex(datasets.ImageFolder):
    def __getitem__(self, index):
        image, label = super().__getitem__(index)
        return image, label, index


def build_class_to_indices(ds) -> Dict[int, List[int]]:
    class_to_indices = defaultdict(list)
    for idx, target in enumerate(ds.targets):
        class_to_indices[target].append(idx)
    return dict(class_to_indices)


# ============================================================
# UA-ProtoNet Model
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        reduced = max(1, in_planes // reduction)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, reduced, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(reduced, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        mask = self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
        self.last_mask = mask
        return x * mask


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        mask = self.sigmoid(self.conv1(torch.cat([avg_out, max_out], dim=1)))
        self.last_mask = mask
        return x * mask


class DualAttention(nn.Module):
    def __init__(self, in_planes, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        self.last_F = x
        return self.sa(self.ca(x))


class EfficientNetB0FSL(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        effnet = models.efficientnet_b0(weights=weights)

        self.features_early = effnet.features[:7]
        self.attention_mid = DualAttention(192, reduction=8)

        self.features_late = effnet.features[7:]
        self.attention_final = DualAttention(1280, reduction=16)

        self.pooling = nn.AdaptiveAvgPool2d(1)

        self.mid_proj = nn.Linear(192, 128)
        self.late_proj = nn.Linear(1280, 128)

        self.mcd_dropout = nn.Dropout(p=0.5)
        self.proj = nn.Linear(256, 256)

        self.temperature = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        x = self.features_early(x)
        x_mid = self.attention_mid(x)

        x = self.features_late(x_mid)
        x_late = self.attention_final(x)

        mid_feat = torch.flatten(self.pooling(x_mid), 1)
        late_feat = torch.flatten(self.pooling(x_late), 1)

        mid_feat = self.mid_proj(mid_feat)
        late_feat = self.late_proj(late_feat)

        combined = torch.cat([mid_feat, late_feat], dim=1)

        combined = self.mcd_dropout(combined)
        combined = self.proj(combined)

        return F.normalize(combined, dim=1)

    def enable_mcdropout(self):
        self.mcd_dropout.train()


# ============================================================
# Regularization Loss Functions & Augmentations
# ============================================================

class ContrastiveLoss(nn.Module):
    def __init__(self, margin: float = 1.0):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        dist_matrix = torch.cdist(embeddings, embeddings, p=2)
        label_eq = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        mask_upper = torch.triu(torch.ones_like(dist_matrix, dtype=torch.bool), diagonal=1)

        pos_pairs = label_eq * dist_matrix
        neg_pairs = (1 - label_eq) * F.relu(self.margin - dist_matrix)

        loss = (pos_pairs[mask_upper].sum() + neg_pairs[mask_upper].sum()) / mask_upper.sum().clamp(min=1)
        return loss


def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    perm = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[perm]
    y_a, y_b = y, y[perm]
    return mixed_x, y_a, y_b, lam


def mixup_cross_entropy(logits, y_a, y_b, lam, label_smoothing=0.1):
    return (
        lam * F.cross_entropy(logits, y_a, label_smoothing=label_smoothing) +
        (1 - lam) * F.cross_entropy(logits, y_b, label_smoothing=label_smoothing)
    )


def cutmix_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.2):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.shape
    perm = torch.randperm(B, device=x.device)

    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = int(np.clip(cx - cut_w // 2, 0, W))
    x2 = int(np.clip(cx + cut_w // 2, 0, W))
    y1 = int(np.clip(cy - cut_h // 2, 0, H))
    y2 = int(np.clip(cy + cut_h // 2, 0, H))

    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]

    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    return mixed_x, y, y[perm], lam


def orthogonal_reg(prototypes, weight=0.01):
    proto_norm = F.normalize(prototypes, dim=1)
    similarity = torch.mm(proto_norm, proto_norm.t())
    identity = torch.eye(similarity.size(0), device=similarity.device)
    return ((similarity - identity) ** 2).sum() * weight


# ============================================================
# Configuration & Multi-Seed Setup
# ============================================================

@dataclass
class Config:
    data_root: str = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
    image_size: int = 224
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    batch_size: int = 1
    num_workers: int = 2

    lr: float = 1e-3
    weight_decay: float = 1e-2
    epochs: int = 50

    train_episodes: int = 600
    val_episodes: int = 300
    test_episodes: int = 600

    n_way: int = 5
    k_shot: int = 5
    q_query: int = 15

    patience: int = 10

    contrastive_margin: float = 1.0
    contrastive_lambda: float = 0.1
    mixup_alpha: float = 0.2
    label_smoothing: float = 0.1
    ortho_weight: float = 0.01
    cutmix_prob: float = 0.5
    warmup_epochs: int = 5

    split_seed: int = 42
    seeds: Tuple[int, ...] = (42, 123, 456)
    mc_dropout_T: int = 10
    eval_seed: int = 20260725

    episode_dir: str = "."
    results_dir: str = "."


CFG = Config()

# ============================================================
# Transforms
# ============================================================

train_tfms = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============================================================
# Fixed Episode Generation / Serialization
# ============================================================

def generate_episodes(
    valid_classes: List[int],
    class_to_indices: Dict[int, List[int]],
    n_way: int,
    k_shot: int,
    q_query: int,
    episodes: int,
    rng: random.Random
) -> List[dict]:
    episode_list = []

    for episode_id in range(episodes):
        sampled_classes = rng.sample(list(valid_classes), n_way)

        support_indices = []
        support_labels = []
        query_indices = []
        query_labels = []

        for mapped_label, cls in enumerate(sampled_classes):
            available = class_to_indices[cls]
            needed = k_shot + q_query

            if len(available) < needed:
                raise ValueError(
                    f"Class {cls} has only {len(available)} images, but needs {needed}."
                )

            selected = rng.sample(available, needed)

            support_indices.extend(selected[:k_shot])
            support_labels.extend([mapped_label] * k_shot)

            query_indices.extend(selected[k_shot:k_shot + q_query])
            query_labels.extend([mapped_label] * q_query)

        episode_list.append({
            "episode_id": episode_id,
            "classes": sampled_classes,
            "support_indices": support_indices,
            "support_labels": support_labels,
            "query_indices": query_indices,
            "query_labels": query_labels
        })

    return episode_list


def load_or_create_episodes(
    split_name: str,
    valid_classes: List[int],
    class_to_indices: Dict[int, List[int]],
    episodes: int,
    rng_seed: int,
    cfg: Config
) -> List[dict]:
    os.makedirs(cfg.episode_dir, exist_ok=True)

    filename = os.path.join(
        cfg.episode_dir,
        f"pv_{split_name}_fixed_episodes_split{cfg.split_seed}_"
        f"n{cfg.n_way}_k{cfg.k_shot}_q{cfg.q_query}_ep{episodes}.json"
    )

    if os.path.exists(filename):
        print(f"Loading fixed {split_name} episodes from: {filename}")
        with open(filename, "r") as f:
            return json.load(f)

    print(f"Creating fixed {split_name} episodes at: {filename}")
    rng = random.Random(rng_seed)

    episode_list = generate_episodes(
        valid_classes=valid_classes,
        class_to_indices=class_to_indices,
        n_way=cfg.n_way,
        k_shot=cfg.k_shot,
        q_query=cfg.q_query,
        episodes=episodes,
        rng=rng
    )

    with open(filename, "w") as f:
        json.dump(episode_list, f)

    return episode_list


# ============================================================
# Episodic Datasets
# ============================================================

class FixedEpisodeDataset(Dataset):
    def __init__(self, base_dataset, episodes: List[dict]):
        self.base_dataset = base_dataset
        self.episodes = episodes

    def __len__(self):
        return len(self.episodes)

    def __getitem__(self, idx):
        ep = self.episodes[idx]

        support_x = torch.stack([
            self.base_dataset[int(i)][0] for i in ep["support_indices"]
        ])
        query_x = torch.stack([
            self.base_dataset[int(i)][0] for i in ep["query_indices"]
        ])

        support_y = torch.tensor(ep["support_labels"], dtype=torch.long)
        query_y = torch.tensor(ep["query_labels"], dtype=torch.long)

        return support_x, support_y, query_x, query_y


class FewShotEpisodeDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        valid_classes,
        class_to_indices,
        n_way,
        k_shot,
        q_query,
        episodes
    ):
        self.base_dataset = base_dataset
        self.valid_classes = valid_classes
        self.class_to_indices = class_to_indices
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.episodes = episodes

    def __len__(self):
        return self.episodes

    def __getitem__(self, _):
        sampled_classes = random.sample(self.valid_classes, self.n_way)

        support_x, support_y = [], []
        query_x, query_y = [], []

        for mapped_label, cls in enumerate(sampled_classes):
            img_indices = random.sample(
                self.class_to_indices[cls],
                self.k_shot + self.q_query
            )

            for i in range(self.k_shot):
                img, _, _ = self.base_dataset[img_indices[i]]
                support_x.append(img)
                support_y.append(mapped_label)

            for i in range(self.k_shot, self.k_shot + self.q_query):
                img, _, _ = self.base_dataset[img_indices[i]]
                query_x.append(img)
                query_y.append(mapped_label)

        return (
            torch.stack(support_x),
            torch.tensor(support_y, dtype=torch.long),
            torch.stack(query_x),
            torch.tensor(query_y, dtype=torch.long)
        )


def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# ============================================================
# Checkpoint Utilities
# ============================================================

def strip_module_prefix(state):
    if not isinstance(state, dict):
        return state

    needs_strip = any(str(k).startswith("module.") for k in state.keys())
    if not needs_strip:
        return state

    cleaned = {}
    for k, v in state.items():
        if str(k).startswith("module."):
            cleaned[str(k)[len("module."):]] = v
        else:
            cleaned[k] = v
    return cleaned


def find_checkpoint(run_seed: int) -> Optional[str]:
    """
    Safe checkpoint finder.

    Only accepts exact checkpoint filenames.
    Does NOT match arbitrary files containing '42', 'opus', etc.
    """
    candidate_names = [
        f"opus_seed_{run_seed}.pth",
        f"opus_seed_{run_seed}.pt",
        f"opus_seed_{run_seed}.ckpt",
    ]

    if run_seed == 42:
        candidate_names += [
            "opus.pth",
            "opus.pt",
            "opus.ckpt",
        ]

    # Deduplicate while preserving priority order.
    seen = set()
    unique_names = []
    for name in candidate_names:
        if name not in seen:
            seen.add(name)
            unique_names.append(name)
    candidate_names = unique_names
    candidate_set = set(candidate_names)

    # ------------------------------------------------------------
    # 1. Current working directory
    # ------------------------------------------------------------
    for name in candidate_names:
        if os.path.isfile(name):
            return name

    # ------------------------------------------------------------
    # 2. Kaggle working directory
    # ------------------------------------------------------------
    working_dir = "/kaggle/working"
    if os.path.isdir(working_dir):
        for name in candidate_names:
            path = os.path.join(working_dir, name)
            if os.path.isfile(path):
                return path

    # ------------------------------------------------------------
    # 3. Kaggle input directory using exact shallow glob patterns
    # ------------------------------------------------------------
    base_input = "/kaggle/input"
    if os.path.isdir(base_input):
        wildcards = [
            "",
            "/*",
            "/*/*",
            "/*/*/*",
            "/*/*/*/*",
            "/*/*/*/*/*",
            "/*/*/*/*/*/*",
            "/*/*/*/*/*/*/*",
        ]

        patterns = []
        for name in candidate_names:
            for wildcard in wildcards:
                patterns.append(f"{base_input}{wildcard}/{name}")

        for pattern in patterns:
            matches = sorted(glob.glob(pattern))
            for match in matches:
                if os.path.isfile(match):
                    return match

        # ------------------------------------------------------------
        # 4. Fallback controlled walk.
        #    Still only accepts exact checkpoint filenames.
        #    Prunes obvious large image dataset branches.
        # ------------------------------------------------------------
        prune_tokens = (
            "plantvillage-dataset",
            "plantdoc-test-train-merged",
            "abdallahalidev/plantvillage",
            "tanvirakon/plantdoc",
        )

        for root, dirs, files in os.walk(base_input):
            root_lower = root.lower()

            if any(token in root_lower for token in prune_tokens):
                dirs[:] = []
                continue

            # Skip hidden directories.
            dirs[:] = [d for d in dirs if not d.startswith(".")]

            for file in files:
                if file in candidate_set:
                    full_path = os.path.join(root, file)
                    if os.path.isfile(full_path):
                        return full_path

    return None


def load_checkpoint_compat(model, path: str, device: str):
    try:
        state = torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(path, map_location=device)
    except Exception:
        state = torch.load(path, map_location=device)

    # If checkpoint is a full model object instead of state dict.
    if isinstance(state, nn.Module):
        state = state.state_dict()

    # Common checkpoint wrappers.
    if isinstance(state, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break

    state = strip_module_prefix(state)
    model.load_state_dict(state)



# ============================================================
# Training Loop
# ============================================================

def train_fsl(model, train_loader, val_loader, cfg: Config, run_seed: int):
    use_cuda = str(cfg.device).startswith("cuda")
    contrastive_fn = ContrastiveLoss(margin=cfg.contrastive_margin)

    head_params = (
        list(model.attention_mid.parameters()) +
        list(model.attention_final.parameters()) +
        list(model.mid_proj.parameters()) +
        list(model.late_proj.parameters()) +
        list(model.proj.parameters()) +
        [model.temperature]
    )

    backbone_params = (
        list(model.features_early.parameters()) +
        list(model.features_late.parameters())
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": head_params, "lr": cfg.lr},
            {"params": backbone_params, "lr": cfg.lr * 0.01}
        ],
        weight_decay=cfg.weight_decay
    )

    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(
                optimizer,
                start_factor=0.01,
                total_iters=cfg.warmup_epochs
            ),
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=cfg.epochs - cfg.warmup_epochs,
                eta_min=1e-6
            )
        ],
        milestones=[cfg.warmup_epochs]
    )

    scaler = make_grad_scaler(use_cuda)

    best_val_acc = -1.0
    epochs_no_improve = 0
    save_filename = f"opus_seed_{run_seed}.pth"

    for epoch in range(cfg.epochs):
        model.train()
        train_loss = 0.0
        train_acc = 0.0

        for support_x, support_y, query_x, query_y in train_loader:
            support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
            support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
            query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
            query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

            optimizer.zero_grad()

            if random.random() < cfg.cutmix_prob:
                mixed_qx, qy_a, qy_b, lam = cutmix_data(
                    query_x, query_y, alpha=cfg.mixup_alpha
                )
            else:
                mixed_qx, qy_a, qy_b, lam = mixup_data(
                    query_x, query_y, alpha=cfg.mixup_alpha
                )

            with autocast_context(cfg.device):
                n_support = support_x.size(0)
                all_emb = model(torch.cat([support_x, mixed_qx], dim=0))

                support_emb = all_emb[:n_support]
                query_emb = all_emb[n_support:]

                prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
                logits = euclidean_logits(query_emb, prototypes, model.temperature)

                ce_loss = mixup_cross_entropy(
                    logits,
                    qy_a,
                    qy_b,
                    lam,
                    label_smoothing=cfg.label_smoothing
                )

                con_loss = contrastive_fn(support_emb, support_y)
                ortho_loss = orthogonal_reg(prototypes, weight=cfg.ortho_weight)

                loss = ce_loss + cfg.contrastive_lambda * con_loss + ortho_loss

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            train_loss += loss.item()
            train_acc += (logits.argmax(1) == qy_a).float().mean().item()

        train_loss /= len(train_loader)
        train_acc /= len(train_loader)

        model.eval()
        val_loss = 0.0
        val_acc = 0.0

        with torch.no_grad():
            for support_x, support_y, query_x, query_y in val_loader:
                support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
                support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
                query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
                query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

                with autocast_context(cfg.device):
                    n_support = support_x.size(0)
                    all_emb = model(torch.cat([support_x, query_x], dim=0))

                    support_emb = all_emb[:n_support]
                    query_emb = all_emb[n_support:]

                    prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
                    logits = euclidean_logits(query_emb, prototypes, model.temperature)

                    val_loss += F.cross_entropy(logits.float(), query_y).item()

                val_acc += (logits.argmax(1) == query_y).float().mean().item()

        val_loss /= len(val_loader)
        val_acc /= len(val_loader)

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        print(
            f"[Seed {run_seed:4d}] Epoch {epoch+1:02d}/{cfg.epochs} "
            f"[LR: {current_lr:.6f}] | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_filename)
            print(f"  --> Best model saved to '{save_filename}'. Val Acc: {best_val_acc:.4f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= cfg.patience:
                print(f"\nEarly stopping at epoch {epoch+1}. Best Val Acc: {best_val_acc:.4f}")
                break

    load_checkpoint_compat(model, save_filename, cfg.device)
    print(f"\nSeed {run_seed} training complete. Best Val Acc: {best_val_acc:.4f}")

    return model, best_val_acc


# ============================================================
# Evaluation: Deterministic + MC-Dropout Uncertainty
# ============================================================

@torch.no_grad()
def evaluate_plantvillage(
    model,
    test_loader,
    cfg: Config,
    run_seed: int,
    save_csv: bool = True
) -> dict:
    model.eval()

    print(f"[Seed {run_seed}] Deterministic evaluation on fixed PlantVillage test episodes...")
    det_accs = []

    for episode_idx, (support_x, support_y, query_x, query_y) in enumerate(test_loader):
        support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
        support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
        query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
        query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

        with autocast_context(cfg.device):
            n_support = support_x.size(0)
            all_emb = model(torch.cat([support_x, query_x], dim=0))

            support_emb = all_emb[:n_support]
            query_emb = all_emb[n_support:]

            prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
            logits = euclidean_logits(query_emb, prototypes, model.temperature)
            probs = torch.softmax(logits.float(), dim=-1)

        preds = probs.argmax(dim=1)
        acc = (preds == query_y).float().mean().item()
        det_accs.append(acc)

        if (episode_idx + 1) % 100 == 0:
            print(f"  Deterministic episode {episode_idx + 1}/{len(test_loader)}")

    mcd_accs = []
    entropies = []
    mutual_infos = []

    if cfg.mc_dropout_T > 0:
        print(f"[Seed {run_seed}] MC-Dropout evaluation with T={cfg.mc_dropout_T}...")
        model.eval()
        model.enable_mcdropout()

        for episode_idx, (support_x, support_y, query_x, query_y) in enumerate(test_loader):
            support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
            support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
            query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
            query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

            n_support = support_x.size(0)
            probs_list = []

            with autocast_context(cfg.device):
                for _ in range(cfg.mc_dropout_T):
                    all_emb = model(torch.cat([support_x, query_x], dim=0))

                    support_emb = all_emb[:n_support]
                    query_emb = all_emb[n_support:]

                    prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
                    logits = euclidean_logits(query_emb, prototypes, model.temperature)
                    probs = torch.softmax(logits.float(), dim=-1)

                    probs_list.append(probs)

            probs_stack = torch.stack(probs_list, dim=0)
            p_bar = probs_stack.mean(dim=0)
            preds = p_bar.argmax(dim=1)
            acc = (preds == query_y).float().mean().item()

            entropy = -(p_bar * torch.log(p_bar.clamp(min=1e-12))).sum(dim=1)
            per_sample_entropy = -(
                probs_stack * torch.log(probs_stack.clamp(min=1e-12))
            ).sum(dim=2)
            expected_entropy = per_sample_entropy.mean(dim=0)
            mi = (entropy - expected_entropy).clamp(min=0.0)

            mcd_accs.append(acc)
            entropies.append(entropy.mean().item())
            mutual_infos.append(mi.mean().item())

            if (episode_idx + 1) % 50 == 0:
                print(f"  MC-Dropout episode {episode_idx + 1}/{len(test_loader)}")

        model.eval()

    if save_csv:
        os.makedirs(cfg.results_dir, exist_ok=True)
        csv_path = os.path.join(
            cfg.results_dir,
            f"pv_episode_metrics_seed_{run_seed}.csv"
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "seed",
                "episode_id",
                "det_acc",
                "mcd_acc",
                "predictive_entropy",
                "mutual_information"
            ])

            for i, det_acc in enumerate(det_accs):
                if cfg.mc_dropout_T > 0:
                    writer.writerow([
                        run_seed,
                        i,
                        det_acc,
                        mcd_accs[i],
                        entropies[i],
                        mutual_infos[i]
                    ])
                else:
                    writer.writerow([
                        run_seed,
                        i,
                        det_acc,
                        "",
                        "",
                        ""
                    ])

        print(f"Saved episode metrics: {csv_path}")

    det_mean = float(np.mean(det_accs)) if len(det_accs) > 0 else float("nan")

    if cfg.mc_dropout_T > 0:
        mcd_mean = float(np.mean(mcd_accs))
        entropy_mean = float(np.mean(entropies))
        mi_mean = float(np.mean(mutual_infos))
    else:
        mcd_mean = det_mean
        entropy_mean = float("nan")
        mi_mean = float("nan")

    print(f"[Seed {run_seed}] Deterministic Test Acc: {det_mean * 100:.2f}%")
    if cfg.mc_dropout_T > 0:
        print(f"[Seed {run_seed}] MC-Dropout Test Acc:   {mcd_mean * 100:.2f}%")
        print(f"[Seed {run_seed}] Predictive Entropy:    {entropy_mean:.4f}")
        print(f"[Seed {run_seed}] Mutual Information:    {mi_mean:.4f}")

    return {
        "det_acc": det_mean,
        "mcd_acc": mcd_mean,
        "entropy": entropy_mean,
        "mi": mi_mean
    }


# ============================================================
# Student's t Confidence Interval
# ============================================================

def compute_student_t_ci(values: List[float]) -> dict:
    vals = np.asarray(values, dtype=np.float64)
    n = len(vals)

    if n == 0:
        return {
            "mean": float("nan"),
            "ci": float("nan"),
            "std": float("nan"),
            "tcrit": float("nan"),
            "df": 0
        }

    mean = float(np.mean(vals))

    if n == 1:
        return {
            "mean": mean,
            "ci": 0.0,
            "std": 0.0,
            "tcrit": float("nan"),
            "df": 0
        }

    std = float(np.std(vals, ddof=1))
    df = n - 1

    t_table = {
        1: 12.706,
        2: 4.303,
        3: 3.182,
        4: 2.776,
        5: 2.571,
        6: 2.447,
        7: 2.365,
        8: 2.306,
        9: 2.262,
        10: 2.228
    }

    tcrit = t_table.get(df, 1.96)
    ci = tcrit * (std / np.sqrt(n))

    return {
        "mean": mean,
        "ci": ci,
        "std": std,
        "tcrit": tcrit,
        "df": df
    }


# ============================================================
# Summary Reporting
# ============================================================

def summarize_and_save(all_metrics: List[dict], cfg: Config):
    os.makedirs(cfg.results_dir, exist_ok=True)
    seed_csv = os.path.join(cfg.results_dir, "pv_phase1_seed_metrics.csv")

    with open(seed_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "seed",
            "best_val_acc_pct",
            "det_test_acc_pct",
            "mcd_test_acc_pct",
            "predictive_entropy",
            "mutual_information"
        ])

        for m in all_metrics:
            best_val = m.get("best_val_acc", None)
            best_val_pct = "" if best_val is None else best_val * 100.0

            writer.writerow([
                m["seed"],
                best_val_pct,
                m["det_acc"] * 100.0,
                m["mcd_acc"] * 100.0,
                m["entropy"],
                m["mi"]
            ])

    print(f"Saved seed-level metrics: {seed_csv}")

    summary_csv = os.path.join(cfg.results_dir, "pv_phase1_summary.csv")

    metric_specs = [
        ("det_acc", True, "Deterministic Test Accuracy (%)"),
        ("mcd_acc", True, "MC-Dropout Test Accuracy (%)"),
        ("entropy", False, "Predictive Entropy"),
        ("mi", False, "Mutual Information")
    ]

    summary_rows = []

    for key, is_pct, label in metric_specs:
        vals = []

        for m in all_metrics:
            v = m.get(key, None)
            if v is None:
                continue
            try:
                if np.isnan(v):
                    continue
            except TypeError:
                pass

            vals.append(float(v) * 100.0 if is_pct else float(v))

        stats = compute_student_t_ci(vals)

        summary_rows.append([
            label,
            stats["mean"],
            stats["ci"],
            stats["std"],
            stats["df"],
            stats["tcrit"],
            ";".join(f"{v:.6f}" for v in vals)
        ])

    with open(summary_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "metric",
            "mean",
            "student_t_95_ci_half_width",
            "std",
            "df",
            "tcrit",
            "seed_values"
        ])
        writer.writerows(summary_rows)

    print(f"Saved summary metrics: {summary_csv}")

    print("\n" + "=" * 90)
    print("  PHASE 1 IN-DOMAIN SUMMARY: PlantVillage Unseen Test")
    print("=" * 90)

    for m in all_metrics:
        best_val = m.get("best_val_acc", None)
        best_val_str = "NA" if best_val is None else f"{best_val * 100:.2f}%"

        print(
            f"  Seed {m['seed']:5d} | "
            f"Best Val: {best_val_str} | "
            f"Det Test: {m['det_acc'] * 100:.2f}% | "
            f"MCD Test: {m['mcd_acc'] * 100:.2f}% | "
            f"H: {fmt(m['entropy'])} | "
            f"I: {fmt(m['mi'])}"
        )

    print("-" * 90)

    for row in summary_rows:
        label, mean, ci, std, df, tcrit, _ = row
        print(
            f"  {label:35s}: "
            f"{fmt(mean, 2)} ± {fmt(ci, 2)} "
            f"(95% CI, Student's t, df={df}, tcrit={fmt(tcrit, 3)}) | "
            f"SD={fmt(std, 2)}"
        )

    print("=" * 90)


# ============================================================
# Main Execution
# ============================================================

if __name__ == "__main__":
    print("=" * 90)
    print(f"  Phase 1: UA-ProtoNet Multi-Seed PlantVillage Evaluation")
    print(f"  Seeds: {CFG.seeds}")
    print(f"  Fixed class split seed: {CFG.split_seed}")
    print(f"  MC-Dropout T: {CFG.mc_dropout_T}")
    print("=" * 90)

    print(f"Loading dataset from: {CFG.data_root}")

    full_ds = ImageFolderWithIndex(CFG.data_root, transform=train_tfms)
    val_ds = ImageFolderWithIndex(CFG.data_root, transform=val_tfms)

    class_to_indices = build_class_to_indices(full_ds)

    all_classes = list(range(len(full_ds.classes)))
    random.Random(CFG.split_seed).shuffle(all_classes)

    if len(all_classes) < 38:
        raise ValueError(
            f"Expected at least 38 PlantVillage classes for 23/8/7 split, "
            f"found {len(all_classes)}."
        )

    train_classes = all_classes[:23]
    val_classes = all_classes[23:31]
    test_classes = all_classes[31:38]

    print(
        f"Fixed Split (Seed {CFG.split_seed}): "
        f"{len(train_classes)} Train | "
        f"{len(val_classes)} Val | "
        f"{len(test_classes)} Test"
    )

    val_episodes_json = load_or_create_episodes(
        split_name="val",
        valid_classes=val_classes,
        class_to_indices=class_to_indices,
        episodes=CFG.val_episodes,
        rng_seed=CFG.split_seed + 100001,
        cfg=CFG
    )

    test_episodes_json = load_or_create_episodes(
        split_name="test",
        valid_classes=test_classes,
        class_to_indices=class_to_indices,
        episodes=CFG.test_episodes,
        rng_seed=CFG.split_seed + 200002,
        cfg=CFG
    )

    use_cuda = str(CFG.device).startswith("cuda")

    loader_kwargs = dict(
        batch_size=CFG.batch_size,
        num_workers=CFG.num_workers,
        pin_memory=use_cuda,
        persistent_workers=(CFG.num_workers > 0),
        worker_init_fn=worker_init_fn
    )

    all_seed_metrics = []

    for seed_idx, run_seed in enumerate(CFG.seeds):
        print("\n" + "-" * 90)
        print(f"   STARTING SEED {run_seed} ({seed_idx + 1}/{len(CFG.seeds)})")
        print("-" * 90)

        set_seed(run_seed)

        train_task_ds = FewShotEpisodeDataset(
            base_dataset=full_ds,
            valid_classes=train_classes,
            class_to_indices=class_to_indices,
            n_way=CFG.n_way,
            k_shot=CFG.k_shot,
            q_query=CFG.q_query,
            episodes=CFG.train_episodes
        )

        val_task_ds = FixedEpisodeDataset(
            base_dataset=val_ds,
            episodes=val_episodes_json
        )

        test_task_ds = FixedEpisodeDataset(
            base_dataset=val_ds,
            episodes=test_episodes_json
        )

        train_generator = torch.Generator()
        train_generator.manual_seed(run_seed)

        train_loader = DataLoader(
            train_task_ds,
            shuffle=True,
            generator=train_generator,
            **loader_kwargs
        )

        val_loader = DataLoader(
            val_task_ds,
            shuffle=False,
            **loader_kwargs
        )

        test_loader = DataLoader(
            test_task_ds,
            shuffle=False,
            **loader_kwargs
        )

        model = EfficientNetB0FSL(pretrained=True).to(CFG.device)

        checkpoint_path = find_checkpoint(run_seed)

        if checkpoint_path is not None:
            print(f"[SKIP TRAINING] Found existing checkpoint for seed {run_seed}:")
            print(f"  {checkpoint_path}")
            load_checkpoint_compat(model, checkpoint_path, CFG.device)
            best_val_acc = None
        else:
            model, best_val_acc = train_fsl(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                cfg=CFG,
                run_seed=run_seed
            )

        set_seed(CFG.eval_seed)

        metrics = evaluate_plantvillage(
            model=model,
            test_loader=test_loader,
            cfg=CFG,
            run_seed=run_seed,
            save_csv=True
        )

        metrics["seed"] = run_seed
        metrics["best_val_acc"] = best_val_acc

        all_seed_metrics.append(metrics)

        del model
        del train_loader
        del val_loader
        del test_loader

        if use_cuda:
            torch.cuda.empty_cache()

    summarize_and_save(all_seed_metrics, CFG)


  Phase 1: UA-ProtoNet Multi-Seed PlantVillage Evaluation
  Seeds: (42, 123, 456)
  Fixed class split seed: 42
  MC-Dropout T: 10
Loading dataset from: /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
Fixed Split (Seed 42): 23 Train | 8 Val | 7 Test
Creating fixed val episodes at: ./pv_val_fixed_episodes_split42_n5_k5_q15_ep300.json
Creating fixed test episodes at: ./pv_test_fixed_episodes_split42_n5_k5_q15_ep600.json

------------------------------------------------------------------------------------------
   STARTING SEED 42 (1/3)
------------------------------------------------------------------------------------------
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 216MB/s]


[SKIP TRAINING] Found existing checkpoint for seed 42:
  /kaggle/input/models/babatanvir/opus-42/pytorch/default/1/opus_seed_42.pth
[Seed 42] Deterministic evaluation on fixed PlantVillage test episodes...
  Deterministic episode 100/600
  Deterministic episode 200/600
  Deterministic episode 300/600
  Deterministic episode 400/600
  Deterministic episode 500/600
  Deterministic episode 600/600
[Seed 42] MC-Dropout evaluation with T=10...
  MC-Dropout episode 50/600
  MC-Dropout episode 100/600
  MC-Dropout episode 150/600
  MC-Dropout episode 200/600
  MC-Dropout episode 250/600
  MC-Dropout episode 300/600
  MC-Dropout episode 350/600
  MC-Dropout episode 400/600
  MC-Dropout episode 450/600
  MC-Dropout episode 500/600
  MC-Dropout episode 550/600
  MC-Dropout episode 600/600
Saved episode metrics: ./pv_episode_metrics_seed_42.csv
[Seed 42] Deterministic Test Acc: 95.04%
[Seed 42] MC-Dropout Test Acc:   94.79%
[Seed 42] Predictive Entropy:    0.9534
[Seed 42] Mutual Information:    

# b

In [4]:
import os
import random
import json
import glob
import contextlib
import csv
import hashlib
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

# ============================================================
# Shared Utilities & Seed Setting
# ============================================================

def set_seed(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def autocast_context(device: str):
    if str(device).startswith("cuda"):
        try:
            return torch.amp.autocast(device_type="cuda")
        except AttributeError:
            return torch.cuda.amp.autocast()
    return contextlib.nullcontext()


def safe_float(x):
    if x is None:
        return None
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    if np.isnan(v):
        return None
    return v


def fmt(x, decimals: int = 4) -> str:
    v = safe_float(x)
    if v is None:
        return "NA"
    return f"{v:.{decimals}f}"


def save_json(data, path: str):
    dirname = os.path.dirname(path)
    if dirname:
        os.makedirs(dirname, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


# ============================================================
# ProtoNet Utilities
# ============================================================

def compute_prototypes(embeddings, labels, n_way):
    return torch.stack([embeddings[labels == c].mean(dim=0) for c in range(n_way)])


def euclidean_logits(query_embeddings, prototypes, temperature=1.0):
    return -torch.cdist(query_embeddings, prototypes, p=2) / temperature


# ============================================================
# Dataset helpers
# ============================================================

class ImageFolderWithIndex(datasets.ImageFolder):
    def __getitem__(self, index):
        image, label = super().__getitem__(index)
        return image, label, index


def build_class_to_indices(ds) -> Dict[int, List[int]]:
    class_to_indices = defaultdict(list)
    for idx, target in enumerate(ds.targets):
        class_to_indices[target].append(idx)
    return dict(class_to_indices)


# ============================================================
# UA-ProtoNet Model
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        reduced = max(1, in_planes // reduction)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, reduced, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(reduced, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        mask = self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
        self.last_mask = mask
        return x * mask


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        mask = self.sigmoid(self.conv1(torch.cat([avg_out, max_out], dim=1)))
        self.last_mask = mask
        return x * mask


class DualAttention(nn.Module):
    def __init__(self, in_planes, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        self.last_F = x
        return self.sa(self.ca(x))


class EfficientNetB0FSL(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        effnet = models.efficientnet_b0(weights=weights)

        self.features_early = effnet.features[:7]
        self.attention_mid = DualAttention(192, reduction=8)

        self.features_late = effnet.features[7:]
        self.attention_final = DualAttention(1280, reduction=16)

        self.pooling = nn.AdaptiveAvgPool2d(1)

        self.mid_proj = nn.Linear(192, 128)
        self.late_proj = nn.Linear(1280, 128)

        self.mcd_dropout = nn.Dropout(p=0.5)
        self.proj = nn.Linear(256, 256)

        self.temperature = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        x = self.features_early(x)
        x_mid = self.attention_mid(x)
        x = self.features_late(x_mid)
        x_late = self.attention_final(x)

        mid_feat = torch.flatten(self.pooling(x_mid), 1)
        late_feat = torch.flatten(self.pooling(x_late), 1)

        mid_feat = self.mid_proj(mid_feat)
        late_feat = self.late_proj(late_feat)

        combined = torch.cat([mid_feat, late_feat], dim=1)

        combined = self.mcd_dropout(combined)
        combined = self.proj(combined)

        return F.normalize(combined, dim=1)

    def enable_mcdropout(self):
        self.mcd_dropout.train()


# ============================================================
# Checkpoint Utilities
# ============================================================

def strip_module_prefix(state):
    if not isinstance(state, dict):
        return state

    needs_strip = any(str(k).startswith("module.") for k in state.keys())
    if not needs_strip:
        return state

    cleaned = {}
    for k, v in state.items():
        if str(k).startswith("module."):
            cleaned[str(k)[len("module."):]] = v
        else:
            cleaned[k] = v
    return cleaned


def find_checkpoint(run_seed: int) -> Optional[str]:
    """
    Safe checkpoint finder.

    Only accepts exact checkpoint filenames.
    Does NOT match arbitrary files containing '42', 'opus', etc.
    """
    candidate_names = [
        f"opus_seed_{run_seed}.pth",
        f"opus_seed_{run_seed}.pt",
        f"opus_seed_{run_seed}.ckpt",
    ]

    if run_seed == 42:
        candidate_names += [
            "opus.pth",
            "opus.pt",
            "opus.ckpt",
        ]

    # Deduplicate while preserving priority order.
    seen = set()
    unique_names = []
    for name in candidate_names:
        if name not in seen:
            seen.add(name)
            unique_names.append(name)
    candidate_names = unique_names
    candidate_set = set(candidate_names)

    # ------------------------------------------------------------
    # 1. Current working directory
    # ------------------------------------------------------------
    for name in candidate_names:
        if os.path.isfile(name):
            return name

    # ------------------------------------------------------------
    # 2. Kaggle working directory
    # ------------------------------------------------------------
    working_dir = "/kaggle/working"
    if os.path.isdir(working_dir):
        for name in candidate_names:
            path = os.path.join(working_dir, name)
            if os.path.isfile(path):
                return path

    # ------------------------------------------------------------
    # 3. Kaggle input directory using exact shallow glob patterns
    # ------------------------------------------------------------
    base_input = "/kaggle/input"
    if os.path.isdir(base_input):
        wildcards = [
            "",
            "/*",
            "/*/*",
            "/*/*/*",
            "/*/*/*/*",
            "/*/*/*/*/*",
            "/*/*/*/*/*/*",
            "/*/*/*/*/*/*/*",
        ]

        patterns = []
        for name in candidate_names:
            for wildcard in wildcards:
                patterns.append(f"{base_input}{wildcard}/{name}")

        for pattern in patterns:
            matches = sorted(glob.glob(pattern))
            for match in matches:
                if os.path.isfile(match):
                    return match

        # ------------------------------------------------------------
        # 4. Fallback controlled walk.
        #    Still only accepts exact checkpoint filenames.
        #    Prunes obvious large image dataset branches.
        # ------------------------------------------------------------
        prune_tokens = (
            "plantvillage-dataset",
            "plantdoc-test-train-merged",
            "abdallahalidev/plantvillage",
            "tanvirakon/plantdoc",
        )

        for root, dirs, files in os.walk(base_input):
            root_lower = root.lower()

            if any(token in root_lower for token in prune_tokens):
                dirs[:] = []
                continue

            # Skip hidden directories.
            dirs[:] = [d for d in dirs if not d.startswith(".")]

            for file in files:
                if file in candidate_set:
                    full_path = os.path.join(root, file)
                    if os.path.isfile(full_path):
                        return full_path

    return None


def load_checkpoint_compat(model, path: str, device: str):
    try:
        state = torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(path, map_location=device)
    except Exception:
        state = torch.load(path, map_location=device)

    # If checkpoint is a full model object instead of state dict.
    if isinstance(state, nn.Module):
        state = state.state_dict()

    # Common checkpoint wrappers.
    if isinstance(state, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break

    state = strip_module_prefix(state)
    model.load_state_dict(state)


# ============================================================
# Phase 1 Evaluation Configuration
# ============================================================

@dataclass
class EvalConfig:
    plantvillage_root: str = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
    plantdoc_root: str = "/kaggle/input/datasets/tanvirakon/plantdoc-test-train-merged/combined"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    image_size: int = 224
    batch_size: int = 1
    num_workers: int = 2

    # Cross-domain evaluation protocol.
    # Keep this identical to your baseline PlantDoc protocol.
    # If your baseline used 5-way 5-shot 5-query and 300 episodes, leave as is.
    n_way: int = 5
    k_shot: int = 5
    q_query: int = 5
    eval_episodes: int = 300

    mc_dropout_T: int = 10

    # Fixed PlantVillage class split seed. Must match PlantVillage training code.
    split_seed: int = 42

    # Model training seeds to evaluate.
    eval_seeds: Tuple[int, ...] = (42, 123, 456)

    # Fixed MC-Dropout sampling seed base.
    eval_seed: int = 20260725

    episode_dir: str = "."
    results_dir: str = "."

    # Change this if you modify PD_TO_PV_MAPPING.
    mapping_version: str = "pd_mapping_v1"


ECFG = EvalConfig()

MODE_SEED_OFFSET = {
    "ALL": 1,
    "SEEN": 2,
    "VAL": 3,
    "UNSEEN": 4
}

# ============================================================
# Transforms
# ============================================================

test_tfms = transforms.Compose([
    transforms.Resize((ECFG.image_size, ECFG.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# ============================================================
# Fixed Cross-Domain Episode Generation / Serialization
# ============================================================

def pairs_digest(pairs: List[Tuple[str, int, int]]) -> str:
    items = sorted([(str(name), int(pv), int(pd)) for name, pv, pd in pairs], key=lambda x: x[0])
    s = "|".join(f"{name}:{pv}:{pd}" for name, pv, pd in items)
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:10]


def generate_cross_domain_episodes(
    pairs: List[Tuple[str, int, int]],
    pv_class_to_indices: Dict[int, List[int]],
    pd_class_to_indices: Dict[int, List[int]],
    n_way: int,
    k_shot: int,
    q_query: int,
    episodes: int,
    rng: random.Random
) -> List[dict]:
    episode_list = []

    for episode_id in range(episodes):
        sampled_pairs = rng.sample(pairs, n_way)

        support_indices = []
        support_labels = []
        query_indices = []
        query_labels = []

        for mapped_label, (cname, pv_idx, pd_idx) in enumerate(sampled_pairs):
            pv_avail = pv_class_to_indices[pv_idx]
            pd_avail = pd_class_to_indices[pd_idx]

            if len(pv_avail) < k_shot:
                raise ValueError(f"Not enough PlantVillage images for pair {cname}: {len(pv_avail)} < {k_shot}")
            if len(pd_avail) < q_query:
                raise ValueError(f"Not enough PlantDoc images for pair {cname}: {len(pd_avail)} < {q_query}")

            selected_pv = rng.sample(pv_avail, k_shot)
            selected_pd = rng.sample(pd_avail, q_query)

            support_indices.extend(selected_pv)
            support_labels.extend([mapped_label] * k_shot)

            query_indices.extend(selected_pd)
            query_labels.extend([mapped_label] * q_query)

        episode_list.append({
            "episode_id": episode_id,
            "pairs": sampled_pairs,
            "support_indices": support_indices,
            "support_labels": support_labels,
            "query_indices": query_indices,
            "query_labels": query_labels
        })

    return episode_list


def load_or_create_cross_domain_episodes(
    split_name: str,
    pairs: List[Tuple[str, int, int]],
    pv_class_to_indices: Dict[int, List[int]],
    pd_class_to_indices: Dict[int, List[int]],
    episodes: int,
    rng_seed: int,
    cfg: EvalConfig
) -> List[dict]:
    os.makedirs(cfg.episode_dir, exist_ok=True)

    digest = pairs_digest(pairs)

    filename = os.path.join(
        cfg.episode_dir,
        f"pd_cross_{split_name}_{cfg.mapping_version}_pairs{digest}_rng{rng_seed}_"
        f"split{cfg.split_seed}_n{cfg.n_way}_k{cfg.k_shot}_q{cfg.q_query}_ep{episodes}.json"
    )

    if os.path.exists(filename):
        print(f"Loading fixed PlantDoc {split_name} episodes from: {filename}")
        with open(filename, "r") as f:
            return json.load(f)

    print(f"Creating fixed PlantDoc {split_name} episodes at: {filename}")
    rng = random.Random(rng_seed)

    episode_list = generate_cross_domain_episodes(
        pairs=pairs,
        pv_class_to_indices=pv_class_to_indices,
        pd_class_to_indices=pd_class_to_indices,
        n_way=cfg.n_way,
        k_shot=cfg.k_shot,
        q_query=cfg.q_query,
        episodes=episodes,
        rng=rng
    )

    with open(filename, "w") as f:
        json.dump(episode_list, f)

    return episode_list


# ============================================================
# Cross-Domain Episodic Dataset
# ============================================================

class FixedCrossDomainDataset(Dataset):
    def __init__(self, pv_dataset, pd_dataset, episodes: List[dict]):
        self.pv_dataset = pv_dataset
        self.pd_dataset = pd_dataset
        self.episodes = episodes

    def __len__(self):
        return len(self.episodes)

    def __getitem__(self, idx):
        ep = self.episodes[idx]

        support_x = torch.stack([
            self.pv_dataset[int(i)][0] for i in ep["support_indices"]
        ])
        query_x = torch.stack([
            self.pd_dataset[int(i)][0] for i in ep["query_indices"]
        ])

        support_y = torch.tensor(ep["support_labels"], dtype=torch.long)
        query_y = torch.tensor(ep["query_labels"], dtype=torch.long)

        return support_x, support_y, query_x, query_y


def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# ============================================================
# Evaluation & Uncertainty
# ============================================================

@torch.no_grad()
def evaluate_cross_domain(
    model,
    test_loader,
    cfg: EvalConfig,
    run_seed: int,
    mode_name: str,
    save_csv: bool = True,
    mc_seed: Optional[int] = None
) -> dict:

    total_episodes = len(test_loader)

    if total_episodes == 0:
        return {
            "det_acc": float("nan"),
            "mcd_acc": float("nan"),
            "entropy": float("nan"),
            "mi": float("nan")
        }

    # ---------------------------
    # 1. Deterministic evaluation
    # ---------------------------
    model.eval()
    det_accs = []

    print(f"      [{mode_name}] Deterministic evaluation: {total_episodes} episodes")

    for episode_idx, (support_x, support_y, query_x, query_y) in enumerate(test_loader):
        support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
        support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
        query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
        query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

        with autocast_context(cfg.device):
            n_support = support_x.size(0)
            all_emb = model(torch.cat([support_x, query_x], dim=0))

            support_emb = all_emb[:n_support]
            query_emb = all_emb[n_support:]

            prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
            logits = euclidean_logits(query_emb, prototypes, model.temperature)
            probs = torch.softmax(logits.float(), dim=-1)

        preds = probs.argmax(dim=1)
        acc = (preds == query_y).float().mean().item()
        det_accs.append(acc)

        if (episode_idx + 1) % 50 == 0 or (episode_idx + 1) == total_episodes:
            print(f"      [{mode_name}] Deterministic episode {episode_idx + 1}/{total_episodes}")

    # ---------------------------
    # 2. MC-Dropout evaluation
    # ---------------------------
    mcd_accs = []
    entropies = []
    mutual_infos = []

    if cfg.mc_dropout_T > 0:
        if mc_seed is not None:
            set_seed(mc_seed)

        model.eval()
        model.enable_mcdropout()

        print(f"      [{mode_name}] MC-Dropout evaluation: T={cfg.mc_dropout_T}")

        for episode_idx, (support_x, support_y, query_x, query_y) in enumerate(test_loader):
            support_x = support_x.squeeze(0).to(cfg.device, non_blocking=True)
            support_y = support_y.squeeze(0).to(cfg.device, non_blocking=True)
            query_x = query_x.squeeze(0).to(cfg.device, non_blocking=True)
            query_y = query_y.squeeze(0).to(cfg.device, non_blocking=True)

            n_support = support_x.size(0)
            probs_list = []

            with autocast_context(cfg.device):
                for _ in range(cfg.mc_dropout_T):
                    all_emb = model(torch.cat([support_x, query_x], dim=0))

                    support_emb = all_emb[:n_support]
                    query_emb = all_emb[n_support:]

                    prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
                    logits = euclidean_logits(query_emb, prototypes, model.temperature)
                    probs = torch.softmax(logits.float(), dim=-1)
                    probs_list.append(probs)

            probs_stack = torch.stack(probs_list, dim=0)
            p_bar = probs_stack.mean(dim=0)

            preds = p_bar.argmax(dim=1)
            acc = (preds == query_y).float().mean().item()

            entropy = -(p_bar * torch.log(p_bar.clamp(min=1e-12))).sum(dim=1)
            per_sample_entropy = -(
                probs_stack * torch.log(probs_stack.clamp(min=1e-12))
            ).sum(dim=2)
            expected_entropy = per_sample_entropy.mean(dim=0)
            mi = (entropy - expected_entropy).clamp(min=0.0)

            mcd_accs.append(acc)
            entropies.append(entropy.mean().item())
            mutual_infos.append(mi.mean().item())

            if (episode_idx + 1) % 25 == 0 or (episode_idx + 1) == total_episodes:
                running_mcd = float(np.mean(mcd_accs))
                running_h = float(np.mean(entropies))
                running_i = float(np.mean(mutual_infos))
                print(
                    f"      [{mode_name}] MC episode {episode_idx + 1}/{total_episodes} | "
                    f"Running MCD Acc: {running_mcd * 100:.2f}% | "
                    f"H: {running_h:.4f} | I: {running_i:.4f}"
                )

        model.eval()

    # ---------------------------
    # Save episode-level CSV
    # ---------------------------
    if save_csv:
        os.makedirs(cfg.results_dir, exist_ok=True)
        csv_path = os.path.join(
            cfg.results_dir,
            f"pd_{mode_name}_episode_metrics_seed_{run_seed}.csv"
        )

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "seed",
                "mode",
                "episode_id",
                "det_acc",
                "mcd_acc",
                "predictive_entropy",
                "mutual_information"
            ])

            for i, det_acc in enumerate(det_accs):
                if cfg.mc_dropout_T > 0:
                    writer.writerow([
                        run_seed,
                        mode_name,
                        i,
                        det_acc,
                        mcd_accs[i],
                        entropies[i],
                        mutual_infos[i]
                    ])
                else:
                    writer.writerow([
                        run_seed,
                        mode_name,
                        i,
                        det_acc,
                        "",
                        "",
                        ""
                    ])

        print(f"      [{mode_name}] Saved episode metrics: {csv_path}")

    det_mean = float(np.mean(det_accs)) if len(det_accs) > 0 else float("nan")

    if cfg.mc_dropout_T > 0:
        mcd_mean = float(np.mean(mcd_accs))
        entropy_mean = float(np.mean(entropies))
        mi_mean = float(np.mean(mutual_infos))
    else:
        mcd_mean = det_mean
        entropy_mean = float("nan")
        mi_mean = float("nan")

    return {
        "det_acc": det_mean,
        "mcd_acc": mcd_mean,
        "entropy": entropy_mean,
        "mi": mi_mean
    }


# ============================================================
# Student's t Confidence Interval
# ============================================================

def compute_student_t_ci(values: List[float]) -> dict:
    vals = np.asarray(values, dtype=np.float64)
    n = len(vals)

    if n == 0:
        return {
            "mean": float("nan"),
            "ci": float("nan"),
            "std": float("nan"),
            "tcrit": float("nan"),
            "df": 0
        }

    mean = float(np.mean(vals))

    if n == 1:
        return {
            "mean": mean,
            "ci": 0.0,
            "std": 0.0,
            "tcrit": float("nan"),
            "df": 0
        }

    std = float(np.std(vals, ddof=1))
    df = n - 1

    t_table = {
        1: 12.706,
        2: 4.303,
        3: 3.182,
        4: 2.776,
        5: 2.571,
        6: 2.447,
        7: 2.365,
        8: 2.306,
        9: 2.262,
        10: 2.228
    }

    tcrit = t_table.get(df, 1.96)
    ci = tcrit * (std / np.sqrt(n))

    return {
        "mean": mean,
        "ci": ci,
        "std": std,
        "tcrit": tcrit,
        "df": df
    }


# ============================================================
# Summary Reporting
# ============================================================

def save_seed_level_csv(all_metrics: Dict[str, List[dict]], cfg: EvalConfig):
    os.makedirs(cfg.results_dir, exist_ok=True)
    csv_path = os.path.join(cfg.results_dir, "pd_phase1_seed_metrics.csv")

    mode_order = ["ALL", "SEEN", "VAL", "UNSEEN"]
    modes = mode_order + [m for m in all_metrics.keys() if m not in mode_order]

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "mode",
            "seed",
            "det_test_acc_pct",
            "mcd_test_acc_pct",
            "predictive_entropy",
            "mutual_information"
        ])

        for mode in modes:
            for m in all_metrics.get(mode, []):
                det = safe_float(m.get("det_acc"))
                mcd = safe_float(m.get("mcd_acc"))
                ent = safe_float(m.get("entropy"))
                mi = safe_float(m.get("mi"))

                writer.writerow([
                    mode,
                    m.get("seed", ""),
                    "" if det is None else det * 100.0,
                    "" if mcd is None else mcd * 100.0,
                    "" if ent is None else ent,
                    "" if mi is None else mi
                ])

    print(f"Saved seed-level metrics: {csv_path}")


def summarize_cross_domain_results(all_metrics: Dict[str, List[dict]], cfg: EvalConfig):
    save_seed_level_csv(all_metrics, cfg)

    os.makedirs(cfg.results_dir, exist_ok=True)
    summary_csv = os.path.join(cfg.results_dir, "pd_phase1_summary.csv")

    metric_specs = [
        ("det_acc", True, "Det Acc (%)"),
        ("mcd_acc", True, "MCD Acc (%)"),
        ("entropy", False, "Pred Entropy"),
        ("mi", False, "Mutual Info")
    ]

    summary_rows = []

    mode_order = ["ALL", "SEEN", "VAL", "UNSEEN"]
    modes = mode_order + [m for m in all_metrics.keys() if m not in mode_order]

    print("\n" + "=" * 95)
    print(f"  PHASE 1 CROSS-DOMAIN SUMMARY: PlantDoc Evaluation (Seeds: {cfg.eval_seeds})")
    print("=" * 95)

    for mode in modes:
        metrics_list = all_metrics.get(mode, [])
        if not metrics_list:
            continue

        print(f"  --- {mode} SPLIT ---")

        for m in metrics_list:
            seed_val = m.get("seed", "NA")

            det = safe_float(m.get("det_acc"))
            mcd = safe_float(m.get("mcd_acc"))
            ent = safe_float(m.get("entropy"))
            mi = safe_float(m.get("mi"))

            det_s = "NA" if det is None else f"{det * 100:.2f}%"
            mcd_s = "NA" if mcd is None else f"{mcd * 100:.2f}%"

            print(
                f"    Seed {seed_val:>5} | "
                f"Det: {det_s} | "
                f"MCD: {mcd_s} | "
                f"H: {fmt(ent)} | "
                f"I: {fmt(mi)}"
            )

        for key, is_pct, label in metric_specs:
            vals = []

            for m in metrics_list:
                v = safe_float(m.get(key, None))
                if v is None:
                    continue
                vals.append(float(v) * (100.0 if is_pct else 1.0))

            if vals:
                stats = compute_student_t_ci(vals)
                summary_rows.append([
                    mode,
                    label,
                    stats["mean"],
                    stats["ci"],
                    stats["std"],
                    stats["df"],
                    stats["tcrit"],
                    ";".join(f"{v:.6f}" for v in vals)
                ])

                print(
                    f"    Agg {label:18s}: "
                    f"{fmt(stats['mean'], 2)} ± {fmt(stats['ci'], 2)} "
                    f"(SD={fmt(stats['std'], 2)}, df={stats['df']}, tcrit={fmt(stats['tcrit'], 3)})"
                )

        print()

    with open(summary_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "mode",
            "metric",
            "mean",
            "student_t_95_ci_half_width",
            "std",
            "df",
            "tcrit",
            "seed_values"
        ])
        writer.writerows(summary_rows)

    print(f"Saved aggregated cross-domain summary metrics to: {summary_csv}")
    print("=" * 95)


# ============================================================
# Main Execution
# ============================================================

if __name__ == "__main__":
    print("=" * 95)
    print("  Phase 1: UA-ProtoNet Multi-Seed PlantDoc Cross-Domain Evaluation")
    print(f"  Seeds: {ECFG.eval_seeds}")
    print(f"  Fixed class split seed: {ECFG.split_seed}")
    print(f"  Protocol: {ECFG.n_way}-way {ECFG.k_shot}-shot {ECFG.q_query}-query")
    print(f"  Episodes per mode: {ECFG.eval_episodes}")
    print(f"  MC-Dropout T: {ECFG.mc_dropout_T}")
    print("=" * 95)

    os.makedirs(ECFG.results_dir, exist_ok=True)
    save_json(asdict(ECFG), os.path.join(ECFG.results_dir, "pd_phase1_eval_config.json"))

    print("Loading datasets...")
    pv_ds = ImageFolderWithIndex(ECFG.plantvillage_root, transform=test_tfms)
    pd_ds = ImageFolderWithIndex(ECFG.plantdoc_root, transform=test_tfms)

    pv_class_to_indices = build_class_to_indices(pv_ds)
    pd_class_to_indices = build_class_to_indices(pd_ds)

    pv_name_to_idx = {c: i for i, c in enumerate(pv_ds.classes)}
    pd_name_to_idx = {c: i for i, c in enumerate(pd_ds.classes)}

    PD_TO_PV_MAPPING = {
        "Apple_Scab_Leaf":              "Apple___Apple_scab",
        "Apple_rust_leaf":              "Apple___Cedar_apple_rust",
        "Apple_leaf":                   "Apple___healthy",
        "Bell_pepper_leaf":             "Pepper,_bell___healthy",
        "Bell_pepper_leaf_spot":        "Pepper,_bell___Bacterial_spot",
        "Blueberry_leaf":               "Blueberry___healthy",
        "Cherry_leaf":                  "Cherry_(including_sour)___healthy",
        "Corn_Gray_leaf_spot":          "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
        "Corn_leaf_blight":             "Corn_(maize)___Northern_Leaf_Blight",
        "Corn_rust_leaf":               "Corn_(maize)___Common_rust_",
        "grape_leaf":                   "Grape___healthy",
        "grape_leaf_black_rot":         "Grape___Black_rot",
        "Peach_leaf":                   "Peach___healthy",
        "Potato_leaf_early_blight":     "Potato___Early_blight",
        "Potato_leaf_late_blight":      "Potato___Late_blight",
        "Raspberry_leaf":               "Raspberry___healthy",
        "Soyabean_leaf":                "Soybean___healthy",
        "Squash_Powdery_mildew_leaf":   "Squash___Powdery_mildew",
        "Strawberry_leaf":              "Strawberry___healthy",
        "Tomato_Early_blight_leaf":     "Tomato___Early_blight",
        "Tomato_Septoria_leaf_spot":    "Tomato___Septoria_leaf_spot",
        "Tomato_leaf":                  "Tomato___healthy",
        "Tomato_leaf_bacterial_spot":   "Tomato___Bacterial_spot",
        "Tomato_leaf_late_blight":      "Tomato___Late_blight",
        "Tomato_leaf_mosaic_virus":     "Tomato___Tomato_mosaic_virus",
        "Tomato_leaf_yellow_virus":     "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
        "Tomato_mold_leaf":             "Tomato___Leaf_Mold",
    }

    missing_pd = []
    missing_pv = []
    insufficient = []
    usable_common_pairs_raw = []

    for pd_name, pv_name in PD_TO_PV_MAPPING.items():
        if pd_name not in pd_name_to_idx:
            missing_pd.append(pd_name)
            continue

        if pv_name not in pv_name_to_idx:
            missing_pv.append({
                "plantdoc_class": pd_name,
                "plantvillage_class": pv_name
            })
            continue

        pv_idx = pv_name_to_idx[pv_name]
        pd_idx = pd_name_to_idx[pd_name]

        pv_count = len(pv_class_to_indices[pv_idx])
        pd_count = len(pd_class_to_indices[pd_idx])

        if pv_count < ECFG.k_shot or pd_count < ECFG.q_query:
            insufficient.append({
                "plantdoc_class": pd_name,
                "plantvillage_class": pv_name,
                "pv_count": pv_count,
                "pd_count": pd_count,
                "required_k_shot": ECFG.k_shot,
                "required_q_query": ECFG.q_query
            })
            continue

        usable_common_pairs_raw.append((pd_name, pv_idx, pd_idx))

    audit_data = {
        "missing_in_plantdoc": missing_pd,
        "missing_in_plantvillage": missing_pv,
        "insufficient_samples": insufficient,
        "usable_pairs_count_before_split_filter": len(usable_common_pairs_raw)
    }
    save_json(audit_data, os.path.join(ECFG.results_dir, "pd_mapping_audit.json"))

    if len(pv_ds.classes) < 38:
        raise ValueError(
            f"Expected at least 38 PlantVillage classes for 23/8/7 split, found {len(pv_ds.classes)}."
        )

    # Fixed PlantVillage class split. Must match PlantVillage training code.
    pv_all = list(range(len(pv_ds.classes)))
    random.Random(ECFG.split_seed).shuffle(pv_all)

    train_classes = pv_all[:23]
    val_classes = pv_all[23:31]
    test_classes = pv_all[31:38]

    train_set = set(train_classes)
    val_set = set(val_classes)
    test_set = set(test_classes)
    split_class_set = train_set | val_set | test_set

    # Restrict usable pairs to the fixed 23/8/7 split classes.
    usable_common_pairs = sorted(
        [p for p in usable_common_pairs_raw if p[1] in split_class_set],
        key=lambda x: x[0]
    )

    excluded_outside_split = sorted(
        [p for p in usable_common_pairs_raw if p[1] not in split_class_set],
        key=lambda x: x[0]
    )

    seen_pairs = sorted([p for p in usable_common_pairs if p[1] in train_set], key=lambda x: x[0])
    val_pairs = sorted([p for p in usable_common_pairs if p[1] in val_set], key=lambda x: x[0])
    unseen_pairs = sorted([p for p in usable_common_pairs if p[1] in test_set], key=lambda x: x[0])
    all_pairs = sorted(usable_common_pairs, key=lambda x: x[0])

    print("\nClass split counts:")
    print(f"  PlantVillage train classes: {len(train_classes)}")
    print(f"  PlantVillage val classes:   {len(val_classes)}")
    print(f"  PlantVillage test classes:  {len(test_classes)}")
    print(f"  Usable common pairs within split: {len(usable_common_pairs)}")
    print(f"  ALL pairs:    {len(all_pairs)}")
    print(f"  SEEN pairs:   {len(seen_pairs)}")
    print(f"  VAL pairs:    {len(val_pairs)}")
    print(f"  UNSEEN pairs: {len(unseen_pairs)}")

    if excluded_outside_split:
        print(f"  WARNING: {len(excluded_outside_split)} usable pairs excluded because PV class is outside 23/8/7 split.")

    split_info = {
        "split_seed": ECFG.split_seed,
        "mapping_version": ECFG.mapping_version,
        "plantvillage_class_order": pv_ds.classes,
        "plantdoc_class_order": pd_ds.classes,
        "pv_train_classes": train_classes,
        "pv_val_classes": val_classes,
        "pv_test_classes": test_classes,
        "mapping": PD_TO_PV_MAPPING,
        "missing_pd": missing_pd,
        "missing_pv": missing_pv,
        "insufficient": insufficient,
        "excluded_outside_split": excluded_outside_split,
        "pairs": {
            "ALL": all_pairs,
            "SEEN": seen_pairs,
            "VAL": val_pairs,
            "UNSEEN": unseen_pairs
        }
    }

    save_json(split_info, os.path.join(ECFG.results_dir, "pd_phase1_split_info.json"))

    mode_configs = {
        "ALL":    (all_pairs,    ECFG.split_seed + MODE_SEED_OFFSET["ALL"] * 100000),
        "SEEN":   (seen_pairs,   ECFG.split_seed + MODE_SEED_OFFSET["SEEN"] * 100000),
        "VAL":    (val_pairs,    ECFG.split_seed + MODE_SEED_OFFSET["VAL"] * 100000),
        "UNSEEN": (unseen_pairs, ECFG.split_seed + MODE_SEED_OFFSET["UNSEEN"] * 100000),
    }

    mode_loaders = {}
    use_cuda = str(ECFG.device).startswith("cuda")

    loader_kwargs = dict(
        batch_size=ECFG.batch_size,
        num_workers=ECFG.num_workers,
        pin_memory=use_cuda,
        persistent_workers=(ECFG.num_workers > 0),
        worker_init_fn=worker_init_fn
    )

    for mode, (pairs, rng_seed) in mode_configs.items():
        if len(pairs) < ECFG.n_way:
            print(f"Skipping {mode} evaluation: not enough classes ({len(pairs)} < {ECFG.n_way})")
            continue

        episodes_json = load_or_create_cross_domain_episodes(
            split_name=mode,
            pairs=pairs,
            pv_class_to_indices=pv_class_to_indices,
            pd_class_to_indices=pd_class_to_indices,
            episodes=ECFG.eval_episodes,
            rng_seed=rng_seed,
            cfg=ECFG
        )

        ds = FixedCrossDomainDataset(
            pv_dataset=pv_ds,
            pd_dataset=pd_ds,
            episodes=episodes_json
        )

        mode_loaders[mode] = DataLoader(ds, shuffle=False, **loader_kwargs)

    all_seed_metrics = defaultdict(list)

    for seed_idx, run_seed in enumerate(ECFG.eval_seeds):
        checkpoint_path = find_checkpoint(run_seed)

        if not checkpoint_path:
            print(f"\nWARNING: Could not find checkpoint for seed {run_seed}. Skipping!")
            continue

        print("\n" + "-" * 95)
        print(f"   EVALUATING SEED {run_seed} ({seed_idx + 1}/{len(ECFG.eval_seeds)})")
        print(f"   Loading weights from: {checkpoint_path}")
        print("-" * 95)

        model = EfficientNetB0FSL(pretrained=False).to(ECFG.device)
        load_checkpoint_compat(model, checkpoint_path, ECFG.device)

        print(f"   Learned temperature (tau): {model.temperature.item():.4f}")

        for mode, loader in mode_loaders.items():
            mc_seed = ECFG.eval_seed + MODE_SEED_OFFSET.get(mode, 0)
            print(f"\n   Evaluating {mode} split | MC seed: {mc_seed}")

            metrics = evaluate_cross_domain(
                model=model,
                test_loader=loader,
                cfg=ECFG,
                run_seed=run_seed,
                mode_name=mode,
                save_csv=True,
                mc_seed=mc_seed
            )

            metrics["seed"] = run_seed
            metrics["mode"] = mode
            all_seed_metrics[mode].append(metrics)

        del model
        if use_cuda:
            torch.cuda.empty_cache()

    summarize_cross_domain_results(all_seed_metrics, ECFG)

  Phase 1: UA-ProtoNet Multi-Seed PlantDoc Cross-Domain Evaluation
  Seeds: (42, 123, 456)
  Fixed class split seed: 42
  Protocol: 5-way 5-shot 5-query
  Episodes per mode: 300
  MC-Dropout T: 10
Loading datasets...

Class split counts:
  PlantVillage train classes: 23
  PlantVillage val classes:   8
  PlantVillage test classes:  7
  Usable common pairs within split: 27
  ALL pairs:    27
  SEEN pairs:   15
  VAL pairs:    7
  UNSEEN pairs: 5
Creating fixed PlantDoc ALL episodes at: ./pd_cross_ALL_pd_mapping_v1_pairse90005f6eb_rng100042_split42_n5_k5_q5_ep300.json
Creating fixed PlantDoc SEEN episodes at: ./pd_cross_SEEN_pd_mapping_v1_pairs56e21b1e78_rng200042_split42_n5_k5_q5_ep300.json
Creating fixed PlantDoc VAL episodes at: ./pd_cross_VAL_pd_mapping_v1_pairsd249e9cf71_rng300042_split42_n5_k5_q5_ep300.json
Creating fixed PlantDoc UNSEEN episodes at: ./pd_cross_UNSEEN_pd_mapping_v1_pairsb8e63f6b17_rng400042_split42_n5_k5_q5_ep300.json

-----------------------------------------------

# btm